<h1 style="text-align: center">Notebook: El ciclo de entrenamiento</h1>
<h2 style="text-align: center">Optimizadores, learning rate y schedulers</h2>

<p style="text-align: center">Unidad 1 · Fundamentos de IA</p>
<p style="text-align: center">Este cuaderno acompaña la Clase 2 y nos permite experimentar con distintos optimizadores, valores de <em>learning rate</em> y <em>schedulers</em>, visualizando su efecto sobre la función de pérdida y el propio learning rate.</p>

## Objetivos de este notebook

En la Clase 2 vimos que el ciclo de entrenamiento se resume como:

$$
\text{forward} \rightarrow \text{loss} \rightarrow \text{backward} \rightarrow \text{optimizer.step()}
$$

Este notebook busca responder, de forma experimental, tres preguntas:

1. **¿Cómo cambia el entrenamiento según el optimizador utilizado?** (SGD, Momentum, RMSprop, Adam, AdamW)
2. **¿Qué efecto tiene el learning rate ($\eta$) sobre la convergencia?**
3. **¿Cómo modifica un scheduler el learning rate durante el entrenamiento, y qué impacto tiene en la loss?**

Para cada pregunta vamos a entrenar el mismo modelo (partiendo siempre del mismo punto inicial) bajo distintas configuraciones, y vamos a graficar la loss y el learning rate para poder compararlas.

### Comprobar el acceso a la GPU

Si ejecutan **!nvidia-smi** en una celda, podrán comprobar el tipo de hardware al que tienen acceso. Este notebook funciona igual de bien en CPU, ya que el modelo es una red totalmente conectada pequeña.

In [ ]:
!nvidia-smi

### Importaciones

Este notebook usa únicamente PyTorch y torchvision (no MONAI), porque el objetivo es concentrarnos en el ciclo de entrenamiento en sí y no en el preprocesamiento de imágenes médicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Usando device: {device}")

## 1. Datos: FashionMNIST

Usamos FashionMNIST porque es un dataset liviano y rápido de entrenar: esto nos permite repetir el entrenamiento muchas veces con distintas configuraciones sin esperar demasiado.

Como vimos en la Clase 2 (Bloque 7), el `Dataset` y el `DataLoader` resuelven un problema distinto al del modelo: cómo representar, cargar y agrupar los datos en mini-batches.

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

training_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=transform
)
test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=transform
)

batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# traducimos los nombres de clase para que los gráficos queden en español
nombres_clases_es = {
    "T-shirt/top": "Remera",
    "Trouser": "Pantalón",
    "Pullover": "Pulóver",
    "Dress": "Vestido",
    "Coat": "Abrigo",
    "Sandal": "Sandalia",
    "Shirt": "Camisa",
    "Sneaker": "Zapatilla",
    "Bag": "Bolso",
    "Ankle boot": "Botín",
}
class_names = [nombres_clases_es[c] for c in training_data.classes]

print(f"Ejemplos de entrenamiento: {len(training_data)}, ejemplos de test: {len(test_data)}")
print(f"Clases: {class_names}")

### Algunas imágenes de ejemplo

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax in axes.flat:
    idx = np.random.randint(len(training_data))
    img, label = training_data[idx]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. El modelo

Definimos una red totalmente conectada simple:

$$
\hat y = f_\theta(x)
$$

con parámetros $\theta = \{W^{(1)}, b^{(1)}, W^{(2)}, b^{(2)}, W^{(3)}, b^{(3)}\}$.

La función `make_model()` siempre inicializa la red con la misma semilla. Esto es clave para que, al comparar optimizadores o learning rates, todas las corridas partan exactamente del mismo punto inicial en el espacio de parámetros.

In [ ]:
SEED = 2024


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)


def make_model():
    # misma semilla en cada llamada: todas las corridas parten del mismo modelo inicial
    torch.manual_seed(SEED)
    return NeuralNetwork().to(device)

### Diagrama de la arquitectura

Antes de entrenar, visualicemos la arquitectura de `NeuralNetwork`: cuántas neuronas tiene cada capa y cuántos parámetros entrenables ($\theta$) tiene el modelo en total.

In [ ]:
def plot_model_architecture(model, input_size=28 * 28):
    capas = [input_size]
    for capa in model.linear_relu_stack:
        if isinstance(capa, nn.Linear):
            capas.append(capa.out_features)

    fig, ax = plt.subplots(figsize=(10, 3))
    x_positions = np.linspace(0, 1, len(capas))

    for i, (x, n_neuronas) in enumerate(zip(x_positions, capas)):
        ax.add_patch(plt.Rectangle((x - 0.04, 0.3), 0.08, 0.4, facecolor="#e5f3f8", edgecolor="#2f6690"))
        ax.text(x, 0.5, f"{n_neuronas}", ha="center", va="center", fontsize=11)

        if i == 0:
            etiqueta = "Entrada\n(imagen aplanada)"
        elif i == len(capas) - 1:
            etiqueta = "Salida\n(10 clases)"
        else:
            etiqueta = f"Capa oculta {i}\n(ReLU)"
        ax.text(x, 0.15, etiqueta, ha="center", va="top", fontsize=9)

        if i > 0:
            ax.annotate(
                "", xy=(x - 0.045, 0.5), xytext=(x_positions[i - 1] + 0.045, 0.5),
                arrowprops=dict(arrowstyle="->", color="#333"),
            )

    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("Arquitectura del modelo")
    plt.tight_layout()
    plt.show()

    n_parametros = sum(p.numel() for p in model.parameters())
    print(f"Cantidad total de parámetros entrenables: {n_parametros:,}")


plot_model_architecture(make_model())

## 3. Instrumentar el ciclo de entrenamiento

El ciclo de entrenamiento no cambia respecto de lo visto en la Clase 2:

```python
optimizer.zero_grad()
prediction = model(x)
loss = loss_fn(prediction, y)
loss.backward()
optimizer.step()
```

Lo único que agregamos es **instrumentación**: en cada iteración (mini-batch) guardamos la loss y el learning rate actual (`optimizer.param_groups[0]["lr"]`) en un diccionario `history`. Al final de cada época también evaluamos sobre el conjunto de test, para registrar la loss y la accuracy de validación.

La función `run_experiment` encapsula todo esto: entrena durante `epochs` épocas y devuelve el `history` completo de la corrida, listo para graficar.

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(dataloader, model, optimizer, history):
    model.train()
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        # registramos loss y learning rate en cada mini-batch, no solo al final de la época
        history["train_loss_iter"].append(loss.item())
        history["lr_iter"].append(optimizer.param_groups[0]["lr"])


def evaluate(dataloader, model):
    model.eval()
    total_loss, correct = 0.0, 0
    n_batches = len(dataloader)
    n_samples = len(dataloader.dataset)

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    return total_loss / n_batches, correct / n_samples


def run_experiment(label, model, optimizer, epochs=10, scheduler=None, scheduler_mode="epoch"):
    """Entrena `model` y devuelve un historial con loss, lr y métricas de validación por época."""
    history = {
        "label": label,
        "train_loss_iter": [],
        "lr_iter": [],
        "epoch_train_loss": [],
        "epoch_val_loss": [],
        "epoch_val_acc": [],
    }

    for epoch in range(epochs):
        n_iter_antes = len(history["train_loss_iter"])
        train_one_epoch(train_dataloader, model, optimizer, history)

        epoch_loss = float(np.mean(history["train_loss_iter"][n_iter_antes:]))
        val_loss, val_acc = evaluate(test_dataloader, model)

        history["epoch_train_loss"].append(epoch_loss)
        history["epoch_val_loss"].append(val_loss)
        history["epoch_val_acc"].append(val_acc)

        # el scheduler actualiza el lr recién después de completar la época
        if scheduler is not None:
            if scheduler_mode == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        print(
            f"[{label}] epoch {epoch + 1}/{epochs} "
            f"- train_loss: {epoch_loss:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
        )

    return history

### Funciones para graficar

Definimos dos funciones auxiliares para superponer, en un mismo gráfico, la loss o el learning rate de varias corridas (`histories` es una lista de `history`, uno por corrida).

In [ ]:
def plot_loss(histories, key="epoch_train_loss", title="Comparación de loss"):
    plt.figure(figsize=(9, 5))
    for history in histories:
        x = np.arange(1, len(history[key]) + 1)
        plt.plot(x, history[key], marker="o", label=history["label"])
    plt.xlabel("Época" if "epoch" in key else "Iteración")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def plot_lr(histories, title="Evolución del learning rate por iteración"):
    plt.figure(figsize=(9, 5))
    for history in histories:
        x = np.arange(1, len(history["lr_iter"]) + 1)
        plt.plot(x, history["lr_iter"], label=history["label"])
    plt.xlabel("Iteración")
    plt.ylabel("Learning rate")
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

## Bloque 1 · Comparación de optimizadores

Como vimos en la Clase 2, todos los optimizadores usan el gradiente $g_t = \nabla_\theta \mathcal L_B$, pero no todos lo usan de la misma manera:

| Optimizador | Idea principal |
|---|---|
| SGD | usa directamente el gradiente |
| SGD + Momentum | agrega inercia a las actualizaciones |
| RMSprop | adapta la escala usando gradientes cuadrados |
| Adam | combina momentum y escalado adaptativo |
| AdamW | Adam con weight decay desacoplado |

Vamos a entrenar el mismo modelo inicial con cada optimizador, usando un learning rate razonable para cada familia, durante 10 épocas, y a comparar cómo cae la loss de entrenamiento.

In [ ]:
epochs_optimizadores = 10

configs_optimizadores = {
    "SGD": lambda params: torch.optim.SGD(params, lr=1e-2),
    "SGD + Momentum": lambda params: torch.optim.SGD(params, lr=1e-2, momentum=0.9),
    "RMSprop": lambda params: torch.optim.RMSprop(params, lr=1e-3),
    "Adam": lambda params: torch.optim.Adam(params, lr=1e-3),
    "AdamW": lambda params: torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4),
}

historias_optimizadores = []
for nombre, construir_optimizer in configs_optimizadores.items():
    modelo = make_model()
    optimizer = construir_optimizer(modelo.parameters())
    historia = run_experiment(nombre, modelo, optimizer, epochs=epochs_optimizadores)
    historias_optimizadores.append(historia)

In [ ]:
plot_loss(historias_optimizadores, key="epoch_train_loss", title="Loss de entrenamiento por optimizador")

### Qué observar

- **SGD** suele converger más lento y de forma más ruidosa.
- **SGD + Momentum** típicamente acelera la convergencia y suaviza la trayectoria.
- **RMSprop** y **Adam** adaptan el learning rate por parámetro, lo que suele acelerar la convergencia inicial.
- **AdamW** debería comportarse de forma similar a Adam en la loss de entrenamiento; su diferencia (weight decay desacoplado) se nota más en la capacidad de generalización que en la loss de entrenamiento.

Probár cambiar el `lr` dentro de `configs_optimizadores` para ver cómo cambia esta comparación.

## Bloque 2 · Efecto del learning rate

En la Clase 2 vimos que:

$$
\eta \text{ muy chico} \Rightarrow \text{convergencia lenta}
$$

$$
\eta \text{ muy grande} \Rightarrow \text{oscilación o divergencia}
$$

Fijamos el optimizador (Adam) y variamos únicamente el learning rate, para observar estos tres regímenes.

In [ ]:
epochs_lr = 10

configs_lr = {
    "lr = 1e-1 (grande)": 1e-1,
    "lr = 1e-3 (recomendado)": 1e-3,
    "lr = 1e-5 (chico)": 1e-5,
}

historias_lr = []
for nombre, lr in configs_lr.items():
    modelo = make_model()
    optimizer = torch.optim.Adam(modelo.parameters(), lr=lr)
    historia = run_experiment(nombre, modelo, optimizer, epochs=epochs_lr)
    historias_lr.append(historia)

In [ ]:
plot_loss(historias_lr, key="epoch_train_loss", title="Efecto del learning rate sobre la loss (Adam)")

### Qué observar

- Con `lr = 1e-1` es esperable ver una loss inestable, con oscilaciones grandes o incluso divergencia (valores `nan`).
- Con `lr = 1e-3` (el valor por defecto recomendado para Adam) la loss debería bajar de forma sostenida.
- Con `lr = 1e-5` la loss baja muy lentamente: en 10 épocas probablemente no alcanza a converger.

> Si el gráfico muestra valores `nan` para `lr = 1e-1`, es exactamente el fenómeno de divergencia que predice la teoría de la Clase 2.

## Bloque 3 · Learning rate scheduling

El learning rate no tiene por qué mantenerse constante durante todo el entrenamiento. Vamos a comparar, sobre el mismo optimizador base (SGD + Momentum), cuatro configuraciones:

- **Sin scheduler**: `lr` constante durante todo el entrenamiento.
- **StepLR**: reduce el `lr` a la mitad cada 3 épocas.
- **CosineAnnealingLR**: reduce el `lr` siguiendo una curva de coseno hasta la última época.
- **ReduceLROnPlateau**: reduce el `lr` cuando la loss de validación deja de mejorar.

Entrenamos durante 10 épocas para que el efecto de cada scheduler sea visible tanto en la curva de learning rate como en la curva de loss.

In [ ]:
epochs_schedulers = 10
historias_schedulers = []

# Sin scheduler (baseline): lr constante
modelo = make_model()
optimizer = torch.optim.SGD(modelo.parameters(), lr=1e-1, momentum=0.9)
historias_schedulers.append(
    run_experiment("Sin scheduler", modelo, optimizer, epochs=epochs_schedulers)
)

# StepLR: reduce el lr a la mitad cada 3 épocas
modelo = make_model()
optimizer = torch.optim.SGD(modelo.parameters(), lr=1e-1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
historias_schedulers.append(
    run_experiment(
        "StepLR", modelo, optimizer,
        epochs=epochs_schedulers, scheduler=scheduler, scheduler_mode="epoch",
    )
)

# CosineAnnealingLR: reduce el lr siguiendo una curva de coseno hasta T_max
modelo = make_model()
optimizer = torch.optim.SGD(modelo.parameters(), lr=1e-1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_schedulers)
historias_schedulers.append(
    run_experiment(
        "CosineAnnealingLR", modelo, optimizer,
        epochs=epochs_schedulers, scheduler=scheduler, scheduler_mode="epoch",
    )
)

# ReduceLROnPlateau: reduce el lr cuando la loss de validación deja de mejorar
modelo = make_model()
optimizer = torch.optim.SGD(modelo.parameters(), lr=1e-1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
historias_schedulers.append(
    run_experiment(
        "ReduceLROnPlateau", modelo, optimizer,
        epochs=epochs_schedulers, scheduler=scheduler, scheduler_mode="plateau",
    )
)

In [ ]:
plot_loss(historias_schedulers, key="epoch_train_loss", title="Loss de entrenamiento según scheduler")

In [ ]:
def plot_lr_por_epoca(histories):
    plt.figure(figsize=(9, 5))
    for history in histories:
        # tomamos el lr al final de cada época para comparar entre configuraciones
        batches_por_epoca = len(history["lr_iter"]) // len(history["epoch_train_loss"])
        lr_por_epoca = history["lr_iter"][batches_por_epoca - 1 :: batches_por_epoca]
        x = np.arange(1, len(lr_por_epoca) + 1)
        plt.plot(x, lr_por_epoca, marker="o", label=history["label"])
    plt.xlabel("Época")
    plt.ylabel("Learning rate")
    plt.title("Evolución del learning rate por scheduler")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


plot_lr_por_epoca(historias_schedulers)

### Qué observar

- El gráfico de learning rate debería mostrar claramente la forma de cada estrategia: escalones para `StepLR`, una curva suave para `CosineAnnealingLR`, y reducciones puntuales (no necesariamente periódicas) para `ReduceLROnPlateau`.
- Un learning rate decreciente suele permitir "afinar" la solución hacia el final del entrenamiento, cuando pasos grandes ya no son convenientes.

## Cierre

Comparamos, en las tres secciones anteriores, el efecto del optimizador, del learning rate y del scheduler. La siguiente tabla resume la loss y la accuracy de validación al final del entrenamiento en cada configuración probada.

In [ ]:
todas_las_historias = historias_optimizadores + historias_lr + historias_schedulers

print(f"{'Configuración':<28}{'Val loss final':>16}{'Val accuracy final':>20}")
for historia in todas_las_historias:
    print(
        f"{historia['label']:<28}"
        f"{historia['epoch_val_loss'][-1]:>16.4f}"
        f"{historia['epoch_val_acc'][-1]:>20.4f}"
    )

## Síntesis

Ninguno de estos experimentos cambió el ciclo de entrenamiento en sí:

$$
\boxed{
\text{datos} \rightarrow \text{forward} \rightarrow \text{loss} \rightarrow \text{backward} \rightarrow \text{optimizer}
}
$$

Lo que cambió, en cada bloque, fue **cómo se usa el gradiente** para actualizar $\theta$:

- el **optimizador** define la regla de actualización;
- el **learning rate** define la escala de cada paso;
- el **scheduler** define cómo cambia esa escala a lo largo del entrenamiento.

En la próxima clase vamos a estudiar arquitecturas específicas para imágenes (convoluciones, CNNs, U-Net), donde este mismo ciclo de entrenamiento sigue siendo el motor del aprendizaje.